# HW1 · Exercise 9
Igor Meleshin, Skoltech 2026

In [7]:
from pathlib import Path
from time import perf_counter
from urllib.request import urlretrieve

import numpy as np
import pandas as pd

from ari_good import compress_ari, decompress_ari
from nncp_docker import setup_nncp, compress_nncp, decompress_nncp

### (a) Datasets
The original file downloaded from [Source](https://github.com/karpathy/char-rnn/blob/6f9487a6fe5b420b7ca9afb0d7c078e37c1d1b4e/data/tinyshakespeare/input.txt)

In [8]:
# preliminaries
data_dir = Path("datasets")
data_dir.mkdir(exist_ok=True)
work = Path("tmp")
work.mkdir(exist_ok=True)
size = 32 * 1024
url = ("https://raw.githubusercontent.com/karpathy/char-rnn/"
       "6f9487a6fe5b420b7ca9afb0d7c078e37c1d1b4e/data/tinyshakespeare/input.txt")

source = work / "input.txt"
urlretrieve(url, source)

(PosixPath('tmp/input.txt'), <http.client.HTTPMessage at 0x1147db5f0>)

In [9]:
# data loading
original = source.read_bytes()[:size]

rng = np.random.default_rng(20)
shuffled = rng.permutation(np.frombuffer(original, dtype=np.uint8)).tobytes()
repeated = original[:128] * (size // 128)

names = ["original", "shuffled", "repeated"]
for name, data in zip(names, [original, shuffled, repeated]):
    (data_dir / f"{name}.txt").write_bytes(data)

print('original:', original[:150])
print('shuffled:', shuffled[:150])
print('repeated:', repeated[:150])

original: b'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nA'
shuffled: b' atdnuhlmohs\ni  ms , oeasARhg\nne Wa cb \nstt\n .mt uk ioeh.:, \ni.aomrk   \nnn\nmuaussuheoyonebiieg thuH:niaIv h  k\nvrib\nce Ch  tl Lio:MdosfrI r,ns Boeridw'
repeated: b'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to First Citizen:\nBefore '


### (b) Compression
My arithmetic coder writes the input size with `write_size()` and reads it back with `read_size()`. The frequency table is fixed for each file.

NNCP 3.3 uses the default Transformer profile. [NNCP source](https://bellard.org/nncp/)

In [10]:
setup_nncp(work)

$CR=S_{\mathrm{original}}/S_{\mathrm{compressed}}$ uses the complete archive size. Each time is one wall-clock measurement on an my local Apple M4. The Python coder runs natively and NNCP runs under x86-64 emulation (with docker). NNCP is timed inside the container, Docker startup is excluded. Each archive is decoded after compression.

In [11]:
rows = []
for name in names:
    source = data_dir / f"{name}.txt"
    data = source.read_bytes()
    restored = work / "restored.txt"

    for method in ["Arithmetic", "NNCP"]:
        archive = work / f"{name}.{method}"
        if method == "Arithmetic":
            start = perf_counter()
            compress_ari(source, archive)
            seconds = perf_counter() - start
            decompress_ari(archive, restored)
        else:
            seconds = compress_nncp(source, archive)
            decompress_nncp(archive, restored)

        restored.unlink()
        compressed = archive.stat().st_size
        rows.append([name, method, len(data), compressed, len(data) / compressed, seconds])

results = pd.DataFrame(rows, columns=["Dataset", "Method", "Original (B)", "Compressed (B)", "CR", "Time (s)"])
results.round({"CR": 3, "Time (s)": 3})

,Dataset,Method,Original (B),Compressed (B),CR,Time (s)
0,original,Arithmetic,32768,20378,1.608,0.144
1,original,NNCP,32768,17804,1.840,14.300
2,shuffled,Arithmetic,32768,20378,1.608,0.161
3,shuffled,NNCP,32768,21202,1.546,16.940
4,repeated,Arithmetic,32768,19198,1.707,0.172
5,repeated,NNCP,32768,12163,2.694,18.790


### (c) Empirical entropy
For a byte with count $n_b$, its estimated probability is $p_b=n_b/N$. The empirical entropy $H_0=-\sum p_b\log_2 p_b$ measures bits per byte when the model uses only byte frequencies.

In [15]:
rows = []
for name in names:
    data = np.fromfile(data_dir / f"{name}.txt", dtype=np.uint8)
    counts = np.bincount(data, minlength=256)
    p = counts[counts > 0] / data.size
    H = -np.sum(p * np.log2(p))
    rows.append([name, H, data.size * H / 8])

df = pd.DataFrame(rows, columns=["Dataset", "H (bits)", "NH / 8 (bytes)"])
df

,Dataset,H (bits),NH / 8 (bytes)
0,original,4.724138,19350.067300
1,shuffled,4.724138,19350.067300
2,repeated,4.435923,18169.541394


The original and shuffled files both have $H_0=4.7241$ bits. Shuffling preserves every byte count, so the fixed model gives both sequences the same ideal length. The arithmetic archives take 20,378 bytes for the original and 20,379 for the shuffle. Both include 4 bytes for the input size and 1,024 bytes for the counts.

NNCP uses the preceding bytes to predict the next byte. thats why shuffling breaks up words and common letter sequences, and its archive grows from 17,804 to 21,217 bytes.

The repeated file still has $H_0=4.4359$  because its 128-byte block contains many different characters. After the first block, knowing the position in the block determines every byte. The fixed coder cannot use this repetition and takes 19,198 bytes. NNCP captures some of it and takes 12,163 bytes giving the highest compression ratio (2.694). Training and evaluating the Transformer also makes NNCP slower on all three files in this setup. The arithmetic works much more faster and is a more lightweight method.